This submission will serve as a simpler baseline submission for an xgboost model with (slightly) tuned parameters.

# Import and Prepprocessing

In [4]:
import pandas as pd
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.width', 1000)

import os
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier



data_dir = '..'


def preprocess_train(df):
    ## Rename columns
    col_names = {
        'id':'id',
        'Age':'age',
        'Sex':'sex',
        'Chest pain type':'chest_pain_type',
        'BP':'bp',
        'Cholesterol':'cholesterol',
        'FBS over 120':'fbs_over_120',
        'EKG results':'ekg_results',
        'Max HR':'max_hr',
        'Exercise angina':'exercise_angina',
        'ST depression':'st_depression',
        'Slope of ST':'slope_of_st',
        'Number of vessels fluro':'number_of_vessels_fluro',
        'Thallium':'thallium',
        'Heart Disease':'heart_disease'
    }

    df = df.rename(columns=col_names)

    ## Recode target variable column to binary
    df['heart_disease'] = df['heart_disease'].replace({'Presence':1, 'Absence':0}).astype(int)


    ## Drop id column
    drop_cols = ['id']
    df = df.drop(columns=drop_cols)
    return df

def preprocess_test(df):
    ## Rename columns
    col_names = {
        'id':'id',
        'Age':'age',
        'Sex':'sex',
        'Chest pain type':'chest_pain_type',
        'BP':'bp',
        'Cholesterol':'cholesterol',
        'FBS over 120':'fbs_over_120',
        'EKG results':'ekg_results',
        'Max HR':'max_hr',
        'Exercise angina':'exercise_angina',
        'ST depression':'st_depression',
        'Slope of ST':'slope_of_st',
        'Number of vessels fluro':'number_of_vessels_fluro',
        'Thallium':'thallium',
    }

    df = df.rename(columns=col_names)

    ## Recode target variable column to binary
    # df['heart_disease'] = df['heart_disease'].replace({'Presence':1, 'Absence':0}).astype(int)


    ## Drop id column
    # drop_cols = ['id']
    # df = df.drop(columns=drop_cols)
    return df

In [5]:
df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
df = preprocess_train(df)

df_test = pd.read_csv(os.
path.join(data_dir, 'test.csv'))
df_test = preprocess_test(df_test)

df.head()

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,number_of_vessels_fluro,thallium,heart_disease
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1


# Data Partitioning

In [6]:
dep_var_name = 'heart_disease'


X = df.drop([dep_var_name], axis=1)
y = df[dep_var_name]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instance and Fit XGBClassifer

In [8]:
param_optimal = {
    'learning_rate':0.1,
    'n_estimators':140,
    'max_depth':6,
    'min_child_weight':4,
    'gamma':0,
    'subsample':0.65,
    'colsample_bytree':0.75,
    'objective':'binary:logistic',
    'scale_pos_weight':1,
    'seed':42    
}

xgb = XGBClassifier(**param_optimal).fit(X_train, y_train)
xgb_class = xgb.predict(X_test)

In [19]:
xgb_auc = roc_auc_score(y_test, xgb_class)

print(f"AUC score: {xgb_auc}")

AUC score: 0.88517462302454


In [ ]:
# # Set up k-fold cross-validation
# kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# xgb_cv_scores = cross_val_score(xgb, X, y, cv=kfold, scoring='roc_auc')

# print(f"Mean of CV AUC score: {np.mean(xgb_cv_scores)}")

Mean of CV AUC score: 0.9550455667577399


# Create submission 

There was no major preprocessing that needs to be applied to test.csv. Need to pop out id column though

In [ ]:
submission = pd.DataFrame()

submission['id'] = df_test['id']
X2_test = df_test.drop(['id'], axis=1)

In [ ]:
X2_class = xgb.predict(X2_test)
submission['Heart Disease'] = X2_class

In [18]:
# Export submission to csv
submission.to_csv('submission_1.csv', index=False)

# Submisison Score: 0.88382